# Praktisk del

Vi skal oversette norske radiologirapporter til pasientvennlig språk ved hjelp av en språkmodell (Gemini), og se hvordan ulike instruksjoner endrer hva modellen sier.

---


## Før dere starter (2 minutter)

1. **Logg inn med en Google-konto** (personlig Gmail eller HVL-kontoen din). Det er nødvendig for at `google.colab.ai` skal virke.
2. **Valgfritt — lagre din egen kopi** av notatboken (*File → Save a copy in Drive*) hvis dere vil beholde det dere gjør (egne prompter, mottakere, …).
3. **Kjør én og én celle** med `Shift + Enter` (eller play-knappen til venstre på cellen). **Ikke** bruk *Runtime → Run all* — vi går gjennom cellene sammen og diskuterer underveis.

## 1. Modellen vi bruker: Gemini 2.5 Flash

Vi bruker **Gemini 2.5 Flash** fra Google fordi den er gratis i Colab og fungerer fint på norsk. Modellen er snart ett år gammel og **ikke** lenger blant de beste, men holder godt for dagens formål.


## 2. Kom i gang

Google Colab har en innebygget AI-modul. Vi importerer den og lister modellene som er tilgjengelige.


In [ ]:
from google.colab import ai

ai.list_models()

---

### Hjelpefunksjoner

Cellen under definerer noen hjelpefunksjoner vi bruker resten av notatboken. **Dere trenger ikke se på koden** — bare kjør cellen og gå videre.


In [ ]:
#@title Hjelpefunksjoner — kjør denne cellen
# ============================================================
# Hjelpefunksjoner for workshopen
# ============================================================
import json
import re

import requests
from google.colab import ai
from IPython.display import Markdown, display

DEFAULT_MODEL = "google/gemini-2.5-flash"


def ask(user_text, system=None, model_name=DEFAULT_MODEL):
    """Send en melding til modellen og returner svaret som tekst.

    Args:
        user_text: Selve teksten vi sender (f.eks. en radiologirapport).
        system: Valgfri systeminstruksjon (regler/eksempler). Legges foran
            brukerteksten siden google.colab.ai tar én samlet prompt.
        model_name: Hvilken modell vi bruker.
    """
    full = f"{system}\n\n{user_text}" if system else user_text
    return ai.generate_text(full, model_name=model_name)


def vis(svar):
    """Vis modellens svar formatert som markdown."""
    display(Markdown(svar))


def sml_search(term, limit=3):
    """Søk i Store Medisinske Leksikon.

    Returnerer en liste med tupler (oppslagsord, kort_definisjon, url).
    """
    r = requests.get(
        "https://sml.snl.no/api/v1/search",
        params={"query": term, "limit": limit, "offset": 0},
        timeout=10,
    )
    return [
        (h["headword"], h.get("simple", ""), h["article_url"]) for h in r.json()
    ]


def extract_medical_terms(rapport):
    """Be modellen identifisere medisinske fagtermer som en pasient ikke forstår."""
    prompt = (
        "Trekk ut medisinske fagtermer fra denne radiologirapporten som en "
        "pasient sannsynligvis ikke forstår. Returner kun en JSON-liste med "
        'strenger, ingen forklaring. Eksempel: ["pneumothorax", "pleuravæske"]'
    )
    raw = ask(rapport, system=prompt)
    m = re.search(r"\[.*\]", raw, re.DOTALL)
    return json.loads(m.group()) if m else []


def build_sml_context(rapport):
    """Trekk ut fagtermer fra en rapport og slå dem opp i SML.

    Returnerer (liste med termer, markdown-formatert kontekst-streng).
    """
    termer = extract_medical_terms(rapport)
    biter = []
    for t in termer:
        treff = sml_search(t, limit=1)
        if not treff:
            continue
        headword, simple, _ = treff[0]
        if simple:
            biter.append(f"- **{headword}**: {simple}")
    return termer, "\n".join(biter)


print("Hjelpefunksjoner klare.")


### Test at hjelpefunksjonen virker


In [ ]:
svar = ask("Hva er hovedstaden i Norge? Svar med ett ord.")
print(svar)


---

## 3. Hva er en "prompt"?

Når du chatter med ChatGPT eller Gemini-appen, skriver du en melding og får svar. **Den meldingen er en prompt.**

I dag bruker vi to typer prompter samtidig:

```
┌─────────────────────────────────────────────────────────┐
│ Systemprompt (instruksjoner)       │
│ "Du oversetter radiologirapporter til pasientvennlig    │
│  norsk. Bevar usikkerhet. Ikke legg til prognose..."    │
├─────────────────────────────────────────────────────────┤
│ Brukerprompt (innholdet)      │
│ "CT thorax med kontrast. Det påvises..."                │
└─────────────────────────────────────────────────────────┘
                          ↓
                    [ Modellen ]
                          ↓
                       Svar
```

**Systemprompten** er rammen: hvilken rolle modellen har, hva den skal gjøre, hva den skal unngå. Den er ofte usynlig for sluttbrukeren. Når du chatter i Gemini-appen er det Google som har skrevet en systemprompt for deg.

**Brukerprompten** er det konkrete innholdet – i vårt tilfelle radiologirapporten.

Det vi skal gjøre i de neste delene er å skrive **tre forskjellige systemprompter** og se hva som skjer med samme rapport.

---

## 4. Materialet vi jobber med

Tre korte, **fiktive** radiologirapporter. De er skrevet i samme stil som ekte rapporter, men inneholder ingen pasientdata.

In [ ]:
RAPPORT_1 = """CT thorax med intravenøs kontrast.
Det påvises en 18 mm rund fortetning i høyre overlapp, suspekt for malignitet.
Forstørrede lymfeknuter mediastinalt opp til 15 mm.
Ingen pleuravæske. Ingen tegn til lungeemboli."""

RAPPORT_2 = """MR caput.
Det ses en 8 mm hyperintens lesjon periventrikulært på FLAIR-sekvenser,
mest forenlig med kronisk iskemisk forandring.
Ingen patologisk kontrastopptak. Ingen tegn til akutt infarkt eller blødning."""

RAPPORT_3 = """Røntgen thorax.
Hjertestørrelse innen normalområde.
Det ses fortetning basalt i venstre lunge, kan passe med infiltrat.
Anbefaler klinisk korrelasjon. Ingen pneumothorax."""

RAPPORTER = {
    "Rapport 1 (CT thorax)": RAPPORT_1,
    "Rapport 2 (MR caput)": RAPPORT_2,
    "Rapport 3 (Røntgen thorax)": RAPPORT_3,
}

for navn, r in RAPPORTER.items():
    print(f"--- {navn} ---")
    print(r)
    print()

---

## 5. V1 — Naiv prompt

Vi starter med **det enkleste som finnes**: én linje, ingen regler, ingen eksempler.

Slik bruker de aller fleste ChatGPT eller Gemini i utgangspunktet. Du gir bare en kort instruksjon og håper på det beste.

In [ ]:
V1 = "Forklar denne radiologirapporten på et språk en pasient kan forstå."

vis(ask(RAPPORT_1, system=V1))


### Diskusjonsoppgave (5 min, i grupper)

Kjør V1 på de andre rapportene også (kjør cellen under). Les nøye gjennom utdataene og se etter:

1. **Falsk trygghet** – sier modellen at noe er "normalt" eller "ufarlig" når rapporten ikke sier det?
2. **Forsvunnet usikkerhet** – fjerner modellen ord som *mistanke om*, *kan passe med*, *suspekt for*?
3. **Tilført informasjon** – legger modellen til prognose, årsaker eller symptomer som ikke står i rapporten?
4. **Manglende anbefaling** – ignoreres "anbefaler klinisk korrelasjon" og liknende?

Noter ett konkret eksempel per rapport. Disse skal vi prøve å fikse i neste steg.

In [ ]:
# Prøv V1 på de to andre rapportene
vis("### Rapport 2 (MR caput)")
vis(ask(RAPPORT_2, system=V1))


In [ ]:
# Prøv V1 på de to andre rapportene
vis("### Rapport 3 (Røntgen thorax)")
vis(ask(RAPPORT_3, system=V1))

---

## 6. V2 — Strukturert prompt med regler

Nå skriver vi en *detaljert* systemprompt med tydelige regler. Dette er det de fleste tenker på når de hører "prompt engineering": *legg til flere regler til modellen oppfører seg.*

Reglene under er en forenklet versjon av det vi bruker i RadKom-prosjektet.

In [ ]:
V2 = """Du oversetter radiologiske rapporter til pasientvennlig norsk (bokmål).

Regler:
- Skriv på et nivå en pasient uten medisinsk bakgrunn kan forstå.
- Bevar usikkerhet nøyaktig. Hvis rapporten sier "mistanke om", "kan passe med",
  eller "ingen tegn til", skal dette gjenspeiles i oversettelsen.
- Gjengi kun det rapporten eksplisitt nevner. Ikke legg til prognose, mulige
  årsaker, eller symptomer.
- Beskriv aldri funn som "normale" med mindre rapporten sier det eksplisitt.
- Plasser fagbegrepet i parentes etter den enkle forklaringen
  (f.eks. "væske i lungene (lungeødem)").
- Bruk en nøytral, saklig tone i tredjeperson. Ikke skriv "du" eller "din"."""

vis(ask(RAPPORT_1, system=V2))


### Sammenlign V1 og V2

Hva ble bedre? Hva er fortsatt feil?

Et vanlig mønster: V2 fikser de mest åpenbare problemene (falsk trygghet, prognose), men modellen treffer fortsatt feil tone, glemmer parenteser, eller er ujevn i strukturen fra rapport til rapport.

---

## 7. V3 — Få-eksempel-prompt (*few-shot*)

Modellen produserer ofte bedre output når den ser konkrete eksempler på det vi ønsker, enn når den kun får regler å følge. Effekten er sterkest når eksemplene er nøye valgt og kombineres med tydelige instruksjoner.

```
Zero-shot (ingen eksempler) ──►  modellen må gjette hva "god output" betyr
                                  basert på reglene alene

Few-shot (med eksempler)    ──►  modellen ser konkret hva som forventes
                                  og imiterer mønsteret
```

V3 under er V2 + to ferdig løste eksempler.

In [ ]:
V3 = """Du oversetter radiologiske rapporter til pasientvennlig norsk (bokmål).

Regler:
- Skriv på et nivå en pasient uten medisinsk bakgrunn kan forstå.
- Bevar usikkerhet nøyaktig (f.eks. "mistanke om", "kan passe med", "ingen tegn til").
- Gjengi kun det rapporten eksplisitt nevner. Ikke legg til prognose eller årsaker.
- Beskriv aldri funn som "normale" med mindre rapporten sier det eksplisitt.
- Plasser fagbegrepet i parentes etter den enkle forklaringen.
- Bruk en nøytral, saklig tone i tredjeperson.

---
Eksempel 1:

Rapport:
CT abdomen/bekken med intravenøs kontrast. Blindtarmen fremstår utvidet med
tverrmål opp mot 13 mm. Det ses tydelig infiltrasjon i det periappendikulære
fettvevet, samt en appendikolitt basalt i lumen. Ingen tegn til fri væske.

Pasientvennlig oversettelse:
Undersøkelsen av buken viser tegn på betennelse i blindtarmen. Den er forstørret,
og det er irritasjon i fettvevet rundt den. Det ses også en forkalkning
(appendikolitt) nederst i blindtarmen. Det er ikke tegn til væskeansamling i buken.

---
Eksempel 2:

Rapport:
MR columna lumbalis. I nivå L4/L5 påvises et venstresidig paramediant skiveprolaps
som komprimerer venstre L5-rot i recessen. Det er redusert skivehøyde i samme
nivå forenlig med degenerasjon.

Pasientvennlig oversettelse:
MR-bildene av korsryggen viser at skiven mellom 4. og 5. lendevirvel har presset
seg ut mot venstre side (prolaps). Prolapset trykker på en nerve i dette området.
Skiven i samme nivå har redusert høyde, noe som kan passe med slitasjeforandringer
(degenerasjon)."""

vis(ask(RAPPORT_1, system=V3))


### Side om side: V1, V2, V3

Kjør cellen under for å se alle tre i rekkefølge på samme rapport.

In [ ]:
for navn, prompt in [
    ("V1 (naiv — én linje)", V1),
    ("V2 (regler, ingen eksempler)", V2),
    ("V3 (regler + to eksempler)", V3),
]:
    vis(f"## {navn}")
    vis(ask(RAPPORT_1, system=prompt))


### Diskusjonsoppgave (5 min)

- Hvor stor er forskjellen mellom V1 og V2?
- Hvor stor er forskjellen mellom V2 og V3?
- Hva er det eksemplene i V3 fanger som reglene i V2 ikke gjør?

---

## 8. Et enkelt "agent"-trinn: oppslag i SML

Hittil har modellen bare hatt **sin egen treningsdata** å gå på. Hva betyr det? Modellen ble trent på tekst frem til januar 2025. Hvis et fagord er sjeldent, eller modellen rett og slett ikke kjenner den norske medisinske termen godt, vil den bløffe.

**Løsning:** gi modellen tilgang til en ekstern, autoritativ kilde. Da slipper den å gjette.

Dette mønsteret heter **RAG** – *retrieval-augmented generation* – og er kjernen i de fleste "agent"-systemer i dag, inkludert RadKom.

In [ ]:
#@title Vis RAG-pipeline-diagram
# Tegn en enkel arbeidsflyt for RAG-pipelinen
import matplotlib.pyplot as plt
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(11, 2.8))
ax.set_xlim(0, 11)
ax.set_ylim(0, 2.6)
ax.axis("off")

boxes = [
    (0.2, 0.5, 1.9, 1.2, "Rapport", "#E8F0FE", "black"),
    (2.5, 0.5, 1.9, 1.2, "Modellen finner\nfagtermer", "#FCE8E6", "black"),
    (4.8, 0.5, 1.9, 1.2, "Slå opp i\nSML", "#FEF7E0", "black"),
    (7.1, 0.5, 1.9, 1.2, "Definisjoner\nlegges til prompt", "#E6F4EA", "black"),
    (9.4, 0.5, 1.5, 1.2, "Endelig\nsvar", "#4285F4", "white"),
]

for x, y, w, h, text, color, text_color in boxes:
    ax.add_patch(
        patches.FancyBboxPatch(
            (x, y), w, h,
            boxstyle="round,pad=0.05",
            linewidth=1.5, edgecolor="#5F6368", facecolor=color,
        )
    )
    ax.text(x + w / 2, y + h / 2, text,
            ha="center", va="center", fontsize=10, color=text_color)

# Arrows between boxes
for x1, x2 in [(2.1, 2.5), (4.4, 4.8), (6.7, 7.1), (9.0, 9.4)]:
    ax.annotate("", xy=(x2, 1.1), xytext=(x1, 1.1),
                arrowprops=dict(arrowstyle="->", lw=1.8, color="#5F6368"))

ax.text(5.5, 2.2,
        "Et lite \"agent\"-trinn: modellen tar én avgjørelse, kallet utføres, og resultatet mates tilbake",
        ha="center", fontsize=10, style="italic", color="#5F6368")

plt.tight_layout()
plt.show()

### Steg 1: Hent en definisjon fra SML

Først ser vi hva SML-API-et gir oss. Vi søker på *lungeemboli* og henter de to beste treffene.

In [ ]:
for headword, simple, url in sml_search("lungeemboli", limit=2):
    print(f"• {headword}")
    print(f"  Definisjon: {simple}")
    print(f"  URL: {url}")
    print()

### Steg 2: La modellen plukke ut termer fra en rapport

In [ ]:
termer = extract_medical_terms(RAPPORT_1)
print("Modellen mener disse termene trenger forklaring:")
for t in termer:
    print(f"  • {t}")


### Steg 3: Sett det sammen

Nå kjører vi V3 med SML-kontekst som ekstra hjelp.

In [ ]:
#@title Hjelpefunksjon: oversett_med_sml + sammenligning
def oversett_med_sml(rapport, system_prompt):
    """Oversett en rapport med SML-kontekst som ekstra hjelp."""
    termer, kontekst = build_sml_context(rapport)
    melding = f"""Rapport:
{rapport}

SML-kontekst (for å forstå fagtermene):
{kontekst}

Skriv en pasientvennlig oversettelse."""
    return ask(melding, system=system_prompt), termer


vis("## V3 UTEN SML")
vis(ask(RAPPORT_1, system=V3))

vis("## V3 MED SML-oppslag")
oversettelse, brukte_termer = oversett_med_sml(RAPPORT_1, V3)
vis(oversettelse)
vis(f"*Slo opp: {', '.join(brukte_termer)}*")


### Diskusjonsoppgave

- Velger modellen riktige termer? Faller noen viktige ut?
- Blir forklaringene mer presise, eller bare lengre?
- Hva skjer hvis SML-artikkelen er skrevet på et nivå som er mer teknisk enn pasienten klarer? (Dette er et reelt problem i RadKom.)

**Det viktige poenget:** vi har akkurat bygget et veldig lite "agent"-system. Modellen tar én beslutning (hvilke termer er fremmedord?), kaller en ekstern tjeneste (SML), og fletter resultatet inn i et nytt promptkall. De aller fleste kliniske LLM-systemer i dag bygger på dette mønsteret – bare med flere steg og bedre feilhåndtering.

---

## 9. Bonus — Tilpass for ulike mottakere

Samme rapport, men forklart for ulike pasienter. Endre `MOTTAKER`-variabelen og kjør.

In [ ]:
MOTTAKER = "en 70-åring uten medisinsk bakgrunn"  # @param {type:"string"}

V3_med_mottaker = V3 + f"\n\nMottaker: {MOTTAKER}. Tilpass språknivå deretter."

vis(ask(RAPPORT_1, system=V3_med_mottaker))


Prøv ulike mottakere:

- "et barn på 14 år"
- "en person med høyere utdanning, men ikke medisinsk bakgrunn"
- "en pasient som allerede vet at de har lungekreft og kun vil høre om utviklingen"

**Det vanskelige spørsmålet:** Bør et klinisk verktøy egentlig gjøre denne tilpasningen automatisk? Hvem skal velge mottakerprofilen i et reelt system?

---
## 10. Avslutning — Bruk ditt eget verktøy (multimodal)

Nå skal dere gå **ut av notatboken** og bruke ChatGPT, Claude eller Gemini-appen direkte – det dere bruker til vanlig.

Kopier inn bildet under og spør om forklaring. Se hva dere får tilbake.

### Forslag til hvordan dere kan utforske

Start enkelt og se hva modellen gjør av seg selv:

> *"Hva ser du på dette bildet?"*

Prøv så å bruke det dere har lært i dag. Gi modellen en rolle, et tydelig mål og kanskje et eksempel på hvordan svaret skal se ut:

> *"Du er radiograf og skal forklare funn på dette bildet til en pasient uten medisinsk bakgrunn. Skriv to-tre setninger på enkelt norsk."*

Sammenlign de to svarene. Hva endret seg når dere ga modellen mer kontekst?

### En siste påminnelse

Multimodale modeller gjør fortsatt feil. De kan overse funn, finne på ting som ikke er der, eller beskrive noe feil med stor selvsikkerhet. Vær oppmerksomme på dette når dere tester, og bruk samme kritiske blikk som dere har øvd på i dag.

Samtidig blir modellene bedre fra måned til måned. Det dere ser i dag er ikke det dere kommer til å se om et halvt år. Derfor er det viktig å lære seg å bruke dem godt nå, og å fortsette å teste hva nye versjoner faktisk klarer.

---

![Skulderluksasjon - røntgen](https://upload.wikimedia.org/wikipedia/commons/e/ea/Luxation_epaule.PNG)

*Bildekilde: [Luxation epaule – Wikimedia Commons](https://commons.wikimedia.org/wiki/File:Luxation_epaule.PNG), MB, [CC BY-SA 2.5](https://creativecommons.org/licenses/by-sa/2.5/)*